# Publish LoRA Adapters to HuggingFace Hub

Push trained Condition A and Condition B LoRA adapters, then verify they load back correctly.

## 1. Setup

In [4]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

_AO_ROOT = os.path.expanduser("~/ao-robustness/activation_oracles")
os.chdir(_AO_ROOT)

import gc

import torch
from dotenv import load_dotenv
from peft import PeftModel

from nl_probes.sft import push_lora_to_hf
from nl_probes.utils.activation_utils import get_hf_submodule
from nl_probes.utils.common import load_model, load_tokenizer
from nl_probes.utils.eval import run_evaluation, score_eval_responses

load_dotenv(os.path.expanduser("~/.env"))

MODEL_NAME = "Qwen/Qwen3-8B"
MODEL_NAME_STR = MODEL_NAME.split("/")[-1].replace(".", "_")
DTYPE = torch.bfloat16
DEVICE = torch.device("cuda")

CONDITION_A_LOCAL = f"checkpoints_baseline_cls_{MODEL_NAME_STR}/final"
CONDITION_B_LOCAL = f"checkpoints_openended_squad_{MODEL_NAME_STR}/step_5000"

# --- Set your Hub repo IDs here ---
CONDITION_A_REPO = "japhba/ao-condition-a-qwen3-8b"
CONDITION_B_REPO = "japhba/ao-condition-b-qwen3-8b"

PRIVATE = False

## 2. Login

In [5]:
import huggingface_hub

huggingface_hub.login(token=os.environ["HF_TOKEN"])
info = huggingface_hub.whoami()
print(f"Logged in as: {info['name']}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in as: japhba


## 3. Push Condition A

In [6]:
tokenizer = load_tokenizer(MODEL_NAME)
model = load_model(MODEL_NAME, DTYPE)
model = PeftModel.from_pretrained(model, CONDITION_A_LOCAL, is_trainable=False)
model.print_trainable_parameters()

push_lora_to_hf(
    model=model,
    tokenizer=tokenizer,
    repo_id=CONDITION_A_REPO,
    private=PRIVATE,
    commit_message="Condition A: baseline classification LoRA",
)
print(f"\nCondition A published: https://huggingface.co/{CONDITION_A_REPO}")

📦 Loading tokenizer...
🧠 Loading model...


Loading checkpoint shards: 100%|██████████| 5/5 [00:22<00:00,  4.55s/it]


trainable params: 0 || all params: 8,365,323,264 || trainable%: 0.0000
Pushing LoRA adapter to Hugging Face Hub: japhba/ao-condition-a-qwen3-8b


Processing Files (1 / 1): 100%|██████████|  698MB /  698MB, 69.8MB/s  
New Data Upload: 100%|██████████|  698MB /  698MB, 69.8MB/s  
Processing Files (1 / 1): 100%|██████████| 11.4MB / 11.4MB, 19.1MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


Copying config.json from original model: Qwen/Qwen3-8B
Successfully copied config.json from Qwen/Qwen3-8B
Creating README with base model metadata...
Successfully uploaded README with base model metadata
Successfully pushed LoRA adapter to: https://huggingface.co/japhba/ao-condition-a-qwen3-8b

Condition A published: https://huggingface.co/japhba/ao-condition-a-qwen3-8b


## 4. Push Condition B

In [7]:
# Swap adapter in-place — no need to reload base model
del model
torch.cuda.empty_cache()
gc.collect()

model = load_model(MODEL_NAME, DTYPE)
model = PeftModel.from_pretrained(model, CONDITION_B_LOCAL, is_trainable=False)
model.print_trainable_parameters()

push_lora_to_hf(
    model=model,
    tokenizer=tokenizer,
    repo_id=CONDITION_B_REPO,
    private=PRIVATE,
    commit_message="Condition B: open-ended SQuAD LoRA",
)
print(f"\nCondition B published: https://huggingface.co/{CONDITION_B_REPO}")

🧠 Loading model...


Loading checkpoint shards: 100%|██████████| 5/5 [00:13<00:00,  2.71s/it]


trainable params: 0 || all params: 8,365,323,264 || trainable%: 0.0000
Pushing LoRA adapter to Hugging Face Hub: japhba/ao-condition-b-qwen3-8b


Processing Files (1 / 1): 100%|██████████|  698MB /  698MB, 72.7MB/s  
New Data Upload: 100%|██████████|  698MB /  698MB, 72.7MB/s  
Processing Files (1 / 1): 100%|██████████| 11.4MB / 11.4MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


Copying config.json from original model: Qwen/Qwen3-8B
Successfully copied config.json from Qwen/Qwen3-8B
Creating README with base model metadata...
Successfully uploaded README with base model metadata
Successfully pushed LoRA adapter to: https://huggingface.co/japhba/ao-condition-b-qwen3-8b

Condition B published: https://huggingface.co/japhba/ao-condition-b-qwen3-8b


## 5. Cleanup

In [8]:
del model
torch.cuda.empty_cache()
gc.collect()
print("GPU memory freed.")

GPU memory freed.


## 6. Load from Hub

In [9]:
tokenizer = load_tokenizer(MODEL_NAME)
model = load_model(MODEL_NAME, DTYPE)
model = PeftModel.from_pretrained(model, CONDITION_A_REPO, is_trainable=False)
print(f"Loaded adapter from Hub: {CONDITION_A_REPO}")
print(f"Active adapters: {model.active_adapters}")
model.print_trainable_parameters()

📦 Loading tokenizer...
🧠 Loading model...


Loading checkpoint shards:  40%|████      | 2/5 [00:10<00:15,  5.20s/it]


KeyboardInterrupt: 

## 7. Sanity Check

Run a small classification eval with both the local checkpoint and the Hub adapter to confirm identical outputs.

In [ ]:
from nl_probes.dataset_classes.act_dataset_manager import DatasetLoaderConfig
from nl_probes.dataset_classes.classification import ClassificationDatasetConfig, ClassificationDatasetLoader

HOOK_LAYER = 1
LAYER_PERCENTS = [25, 50, 75]
STEERING_COEFFICIENT = 1.0
GENERATION_KWARGS = {"do_sample": False, "temperature": 0.0, "max_new_tokens": 20}
EVAL_BATCH_SIZE = 128

# Build a small test set
sanity_loader = ClassificationDatasetLoader(
    dataset_config=DatasetLoaderConfig(
        custom_dataset_params=ClassificationDatasetConfig(
            classification_dataset_name="sst2",
            max_end_offset=-3, min_end_offset=-3,
            max_window_size=1, min_window_size=1,
        ),
        num_train=0, num_test=50,
        splits=["test"],
        model_name=MODEL_NAME,
        layer_percents=LAYER_PERCENTS,
        save_acts=True,
        batch_size=EVAL_BATCH_SIZE,
    ),
)
sanity_data = sanity_loader.load_dataset("test")
print(f"Sanity eval set: {len(sanity_data)} examples")

In [ ]:
model.eval()
submodule = get_hf_submodule(model, HOOK_LAYER)

# Eval with Hub adapter (already loaded)
hub_results = run_evaluation(
    eval_data=sanity_data,
    model=model,
    tokenizer=tokenizer,
    submodule=submodule,
    device=DEVICE,
    dtype=DTYPE,
    global_step=-1,
    lora_path=None,
    eval_batch_size=EVAL_BATCH_SIZE,
    steering_coefficient=STEERING_COEFFICIENT,
    generation_kwargs=GENERATION_KWARGS,
)
hub_fmt, hub_acc = score_eval_responses(hub_results, sanity_data)
print(f"Hub adapter — format: {hub_fmt:.3f}, accuracy: {hub_acc:.3f}")

# Eval with local checkpoint
local_results = run_evaluation(
    eval_data=sanity_data,
    model=model,
    tokenizer=tokenizer,
    submodule=submodule,
    device=DEVICE,
    dtype=DTYPE,
    global_step=-1,
    lora_path=CONDITION_A_LOCAL,
    eval_batch_size=EVAL_BATCH_SIZE,
    steering_coefficient=STEERING_COEFFICIENT,
    generation_kwargs=GENERATION_KWARGS,
)
local_fmt, local_acc = score_eval_responses(local_results, sanity_data)
print(f"Local checkpoint — format: {local_fmt:.3f}, accuracy: {local_acc:.3f}")

# Compare outputs
hub_responses = [r.api_response for r in hub_results]
local_responses = [r.api_response for r in local_results]
match = sum(h == l for h, l in zip(hub_responses, local_responses))
print(f"\nExact response match: {match}/{len(hub_responses)} ({match/len(hub_responses)*100:.1f}%)")
assert match == len(hub_responses), "Hub and local responses differ!"

In [ ]:
del model, submodule
torch.cuda.empty_cache()
gc.collect()
print("Done.")